<a href="https://colab.research.google.com/github/boemer00/courses/blob/main/generate_text_rnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generating Shakespeare Text with a Character-based RNN

In [2]:
import numpy as np
import pathlib
from tensorflow.keras.utils import get_file
from tensorflow.keras.layers import TextVectorization

In [3]:
url = 'https://homl.info/shakespeare'
filepath = get_file('shakespeare.txt', url)

with open(filepath) as f:
    text = f.read()

print(len(text))

1115394/1115394 [==============================] - 0s 0us/step
1115394


In [4]:
# display text
text[:100]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

## Encode Text as Vectors

In [5]:
# split based on characters
text_vec_layer = TextVectorization(split='character',
                                   standardize='lower')

# fit the text vectorizer on our text
text_vec_layer.adapt([text])
encoded = text_vec_layer([text])[0]

In [6]:
print(f"Shakespeare characters:\n{text[:13]}")
print(f"Encoded text:\n{encoded[:13]}")
print(f"Note the 'i's, represented by 7's in the 2nd, 8th and 10th positions")

Shakespeare characters:
First Citizen
Encoded text:
[21  7 10  9  4  2 20  7  4  7 37  3 11]
Note the 'i's, represented by 7's in the 2nd, 8th and 10th positions


Now, we need a function that will generate a `tf.data.Dataset` from the input sequence of integers.

We need to:

- have a sliding window
- batch the windows into batches
- shuffle those samples (note we can do this here, even though it's a time series forecasting task, because our window includes our label; so our data is alread in X-y form)
- generate tuples of (X, y) pairs from our shuffled batches

In [7]:
import tensorflow as tf

In [8]:
def to_dataset(sequence, length, shuffle=False, seed=None, batch_size=32, prefetch=True):
    """
    Given a sequence of tensors (e.g., a sequence of characters or words in a text dataset),
    returns a dataset of sliding windows of that sequence. Each window contains `length + 1` elements,
    where the first `length` elements are the features (X) and the last element is the label (y).

    Args:
        sequence (Tensor): The input sequence of data.
        length (int): The length of the sliding window (not including the label).
        shuffle (bool, optional): If True, the windows will be shuffled randomly. Defaults to False.
        seed (int, optional): A seed for the random number generator (if shuffling). Defaults to None.
        batch_size (int, optional): The number of windows in each training batch. Defaults to 32.
        prefetch (bool, optional): If True, activates prefetching for improved performance. Defaults to True.

    Returns:
        tf.data.Dataset: A dataset containing batches of sliding windows from the input sequence.
    """
    # Create a tf.data.Dataset from the input Tensor sequence
    dataset = tf.data.Dataset.from_tensor_slices(sequence)

    # Create a sliding window iterator over the sequence, where each window includes 'length + 1' elements
    # The '+ 1' is to include the label (next element in the sequence) in each window
    dataset = dataset.window(length + 1, shift=1, drop_remainder=True)

    # Convert each window into a batch to create single tensors from the elements in the window
    # This ensures that each window is represented as a single tensor of shape `(length + 1,)`
    dataset = dataset.flat_map(lambda window: window.batch(length + 1))

    # Shuffle the windows if the 'shuffle' parameter is True
    if shuffle:
        dataset = dataset.shuffle(buffer_size=100_000, seed=seed)

    # Group the windows into training batches of the specified size
    dataset = dataset.batch(batch_size)

    # Split each window into features (X) and label (y), then activate prefetching if specified
    # The features are the first 'length' elements of the window, and the label is the last element
    if prefetch:
        return dataset.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1)
    else:
        return dataset.map(lambda window: (window[:, :-1], window[:, 1:]))


In [9]:
# set the window length
length = 100

# set a random seed
seed=42
tf.random.set_seed(seed)

# generate iterables from the encoded text
train_set = to_dataset(encoded[:1_000_000], length=length, shuffle=True, seed=seed)
valid_set = to_dataset(encoded[1_000_000:1_060_000], length=length)
test_set = to_dataset(encoded[1_060_000:], length=length)

# Text Classification with Transformer

In [10]:
from keras.models import load_model, Sequential
from keras.layers import Embedding, LSTM, GRU, Dense
from keras.callbacks import ModelCheckpoint
import pathlib

In [11]:
%%time

n_tokens = text_vec_layer.vocabulary_size()
path_to_saved_character_rnn = pathlib.Path('shakespeare_model')

if path_to_saved_character_rnn.exists():
  model = load_model(path_to_saved_character_rnn)
else:
  model = Sequential()
  model.add(Embedding(input_dim=n_tokens, output_dim=16))
  model.add(LSTM(256, return_sequences=True))
  model.add(Dense(n_tokens, activation='softmax'))

  model.compile(
      loss='sparse_categorical_crossentropy',
      optimizer='nadam',
      metrics=['accuracy']
  )
  model_ckpt = ModelCheckpoint(
      'shakespeare_model',
      monitor='val_accuracy',
      save_best_only=True
  )
  history = model.fit(
      train_set,
      validation_data=valid_set,
      epochs=2,
      callbacks=[model_ckpt]
  )
  model.save(path_to_saved_character_rnn)

Epoch 1/2
  31247/Unknown - 12842s 410ms/step - loss: 1.2455 - accuracy: 0.6200

31247/31247 [==============================] - 13079s 417ms/step - loss: 1.2455 - accuracy: 0.6200 - val_loss: 1.7838 - val_accuracy: 0.5160
Epoch 2/2
31247/31247 [==============================] - ETA: 0s - loss: 1.0705 - accuracy: 0.6683

31247/31247 [==============================] - 13972s 446ms/step - loss: 1.0705 - accuracy: 0.6683 - val_loss: 1.7744 - val_accuracy: 0.5218


CPU times: user 10h 3min 53s, sys: 18min 32s, total: 10h 22min 25s
Wall time: 7h 31min 29s


Now we take the model above, and wrap it into another model that will perform the character embedding step to make it easier for us to make predictions:

In [12]:
wrapper_model = Sequential([
    text_vec_layer,
    model
])

And finally, we can test our model at predicting the next letter of a sequence:

In [13]:
example_sentence = 'To be or not to b'
prediction_probabilities = wrapper_model.predict([example_sentence])[0, -1]
prediction = tf.argmax(prediction_probabilities)

print(f'Input sentence: {example_sentence}')
print(f'Predicted character: {text_vec_layer.get_vocabulary()[prediction]}')

1/1 [==============================] - 1s 625ms/step
Input sentence: To be or not to b
Predicted character: e


That was the correct prediction. Let's predict a bit more of Shakespeare-like text.

In [14]:
def predict_next_character(text, temperature=1):
    probabilities = wrapper_model.predict([text])[0, -1:]
    rescaled_logits = tf.math.log(probabilities) / temperature
    char_id = tf.random.categorical(rescaled_logits, num_samples=1)[0, 0]
    return text_vec_layer.get_vocabulary()[char_id]

def generate_text(text, n_chars=100, temperature=1):
    for _ in range(n_chars):
        text += predict_next_character(text, temperature)
    return text

In [15]:
tf.random.set_seed(42)

print(generate_text('To be or not to be', temperature=0.1))

1/1 [==============================] - 0s 71ms/step
To be or not to be a storm
that maid of the seal of sight of here,
and therefore from me to be a thief with the strong


# Save Model

In [3]:
print(pathlib.Path('shakespeare_model').resolve())

/content/shakespeare_model


In [4]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define the save path
save_path = "/content/drive/My Drive/your_directory/shakespeare_model.h5"

# Save the model to the specified path
model.save(save_path)

print(f"Model has been saved to {save_path}")


Mounted at /content/drive


NameError: ignored